In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
KEY = os.getenv("CENSUS_API_KEY")

YEARS = [2019, 2024]   # 2019 = the 2015–2019 survey; 2024 = the 2020–2024 survey
STATE = "24"           # Maryland's FIPS code

In [2]:
DETAILED_VARS = {
    "B19013_001E": "median_hh_income",
    "B17001_001E": "pov_universe",
    "B17001_002E": "pov_below",
    "B25070_001E": "renters_total",
    "B25070_007E": "rent_30_35",
    "B25070_008E": "rent_35_40",
    "B25070_009E": "rent_40_50",
    "B25070_010E": "rent_50_plus",
    "B25070_011E": "rent_not_computed",
    "B15003_001E": "pop_25_plus",
    "B15003_022E": "edu_bachelors",
    "B15003_023E": "edu_masters",
    "B15003_024E": "edu_professional",
    "B15003_025E": "edu_doctorate",
    "B28002_001E": "households_total",
    "B28002_004E": "hh_broadband",
}

SUBJECT_VARS = {
    "S2701_C05_001E": "pct_uninsured",
}

In [3]:
def check_labels(year, variables, dataset="acs/acs5"):
    for code in variables:
        url = f"https://api.census.gov/data/{year}/{dataset}/variables/{code}.json"
        label = requests.get(url).json()["label"]
        print(f"{year} | {code} | {label}")

for year in YEARS:
    check_labels(year, DETAILED_VARS)
    check_labels(year, SUBJECT_VARS, dataset="acs/acs5/subject")

2019 | B19013_001E | Estimate!!Median household income in the past 12 months (in 2019 inflation-adjusted dollars)
2019 | B17001_001E | Estimate!!Total:
2019 | B17001_002E | Estimate!!Total:!!Income in the past 12 months below poverty level:
2019 | B25070_001E | Estimate!!Total:
2019 | B25070_007E | Estimate!!Total:!!30.0 to 34.9 percent
2019 | B25070_008E | Estimate!!Total:!!35.0 to 39.9 percent
2019 | B25070_009E | Estimate!!Total:!!40.0 to 49.9 percent
2019 | B25070_010E | Estimate!!Total:!!50.0 percent or more
2019 | B25070_011E | Estimate!!Total:!!Not computed
2019 | B15003_001E | Estimate!!Total:
2019 | B15003_022E | Estimate!!Total:!!Bachelor's degree
2019 | B15003_023E | Estimate!!Total:!!Master's degree
2019 | B15003_024E | Estimate!!Total:!!Professional school degree
2019 | B15003_025E | Estimate!!Total:!!Doctorate degree
2019 | B28002_001E | Estimate!!Total:
2019 | B28002_004E | Estimate!!Total:!!With an Internet subscription!!Broadband of any type
2019 | S2701_C05_001E | Est

In [4]:
def get_acs(year, variables, dataset="acs/acs5"):
    url = f"https://api.census.gov/data/{year}/{dataset}"
    params = {
        "get": "NAME," + ",".join(variables),
        "for": "county:*",
        "in": f"state:{STATE}",
        "key": KEY,
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    rows = response.json()
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df.rename(columns=variables)

# 01 - Pull Data
Pulls American Community Survey (ACS) 5-year estimates for Maryland's 24 jurisdictions for two periods (2015-2019 and 2020-2024) from the census API, saves them to 'data/raw/'

In [5]:
df_2024 = pd.read_csv("../data/raw/acs_2024.csv")
df_2024.head()

,NAME,median_hh_income,pov_universe,pov_below,renters_total,rent_30_35,rent_35_40,rent_40_50,rent_50_plus,rent_not_computed,...,edu_bachelors,edu_masters,edu_professional,edu_doctorate,households_total,hh_broadband,state,county,pct_uninsured,year
0,"Allegany County, Maryland",59603,61133,10302,8019,574,315,563,1352,1581,...,5613,3486,561,407,27382,23241,24,1,3.8,2024
1,"Anne Arundel County, Maryland",124911,584318,32980,55896,4517,3805,5696,11142,2542,...,104297,64802,11770,8921,224748,214663,24,3,4.7,2024
2,"Baltimore County, Maryland",91768,831275,81318,111893,9914,7289,10320,30055,5617,...,137060,80542,19703,12138,332801,304196,24,5,5.4,2024
3,"Calvert County, Maryland",133922,93877,3632,4282,209,255,411,981,393,...,13292,8634,1100,1113,33465,31319,24,9,3.0,2024
4,"Caroline County, Maryland",68457,33177,4035,3517,595,216,296,689,297,...,2190,1217,198,128,12495,11032,24,11,7.3,2024


In [6]:
for year in YEARS:
    detailed = get_acs(year, DETAILED_VARS)
    subject = get_acs(year, SUBJECT_VARS, dataset="acs/acs5/subject")

    df = detailed.merge(subject[["state", "county", "pct_uninsured"]], on=["state", "county"])
    df["year"] = year

    path = f"../data/raw/acs_{year}.csv"
    df.to_csv(path, index=False)
    print(f"Saved {path}: {df.shape[0]} rows, {df.shape[1]} columns")

Saved ../data/raw/acs_2019.csv: 24 rows, 21 columns
Saved ../data/raw/acs_2024.csv: 24 rows, 21 columns
